In [ ]:
import cv2

# imread: 디스크에 있는 이미지 파일(png, jpg 등)을 읽어서 numpy 배열로 메모리에 올려주는 함수
# 컬러로 읽기 → 채널 3개(B, G, R) 포함된 배열로 반환 (OpenCV는 RGB가 아니라 BGR 순서)
color_img = cv2.imread("Lenna.png", cv2.IMREAD_COLOR)
# 흑백으로 읽기 → 채널 없이 밝기값 1개만 있는 배열로 반환
gray_img = cv2.imread("Lenna.png", cv2.IMREAD_GRAYSCALE)

# (세로 픽셀 수, 가로 픽셀 수, 채널 수) 형태로 출력됨. 예: (512, 512, 3)
print(color_img.shape)
# 채널 정보 없이 (세로, 가로)만 출력됨. 예: (512, 512)
print(gray_img.shape)

# imwrite: 이미지 저장
print("저장 완료") if cv2.imwrite("gray_Lenna.png", gray_img) else print("저장 실패")
# cvtColor: 색깔 바꾸기
gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)

cv2.imwrite("converted_Lenna.png", gray_img)

# resize: 이미지 크기 변경
resized_img = cv2.resize(color_img, (1024, 1024))
print(resized_img.shape)

cropped_image = color_img[:color_img.shape[0] // 2, :color_img.shape[1] // 2]

print(cropped_image.shape)
cv2.imwrite("cropped_Lenna.png", cropped_image)

In [ ]:
# rectangle: 이미지 위에 사각형 그리기 (얼굴 부근 박스)
rect_img = color_img.copy()  # rectangle은 원본을 직접 수정하므로 복사본에 그림
cv2.rectangle(rect_img, (220, 200), (400, 420), (0, 255, 0), 2)  # (좌상단), (우하단), 색(BGR), 두께
cv2.imwrite("rect_Lenna.png", rect_img)

In [ ]:
# 영상 처리 (Image Processing)
# 정의: 입력 영상을 원하는 영상으로 변환하거나, 영상에서 특성(feature)을 추출하는 것
#       → 사람이 이해할 수 있는 형태로 만드는 것

# 1) 저수준 영상 처리 (Low-level)
#    - 입력: 영상 → 출력: 영상
#    - 목적: 사람이 "보기에" 개선됨 (화질 개선, 노이즈 제거, 대비 조정 등)

# 2) 고수준 영상 처리 (High-level)
#    - 입력: 영상 → 출력: 의미 있는 정보 (라벨, 좌표 등)
#    - 목적: 객체/내용을 인식·표현할 수 있음

# 영상의 디지털화 = 표본화 + 양자화
# 표본화(Sampling): 연속된 공간을 일정 간격으로 나눠 이산적인 점(픽셀)으로 만드는 것 → 해상도 결정
# 양자화(Quantization): 각 점의 밝기(연속값)를 정해진 단계의 정수로 바꾸는 것 → 보통 8bit(0~255)

# 회선처리(컨볼루션-convolution): 이미지에 커널을 갖다 대고 슬라이딩하며, 겹치는 원소끼리 곱해서 더한 값을 각 위치마다 하나씩 준다.(그 결과가 피처맵)
# (이미지의 특정 패턴이 어디 있는지 숫자로 나타내서 컴퓨터가 알 수 있게 하기 위해 - 학습에 사용)
# 패딩: 원본 이미지 테두리에 0을 둘러서, 컨볼루션해도 크기가 안 줄어들게 하는 것.

# 스트라이드: 커널 이동 보폭(클수록 출력 작아지고 빠름) - 인식률 저조
# 풀링: 이미지에 특정 영역을 갖다 대서
#   - 맥스풀링: 가장 큰 값(강한 신호)만 남김
#   - 평균풀링: 평균값(대표값)만 남김
#   - 민풀링: 가장 작은 값(약한 신호)만 남김
# 효과 1: 정보의 수를 줄여 추론속도를 높임
# 효과 2: 위치가 살짝 바뀌어도 같은 결과가 나오게 함 (정확한 위치 대신 "여기 신호 있었다"만 남기므로)

# 학습 순서: 회선처리(특징 추출) -> 풀링(차원 축소) -> 회선처리 -> 풀링

In [ ]:
# 원본 이미지 → OpenCV(전처리: 리사이즈, 정규화, 크롭 등) → CNN(특징 추출 + 분류/탐지) → 결과

# CNN 네트워크 디자인
# 입력 이미지
#   → 컨볼루션 + ReLU             : 특징 추출 + 음수 없애서 비선형성 추가
#   → 풀링                       : 크기 축소, 위치 변화에 강건하게
#   → 컨볼루션 + ReLU             : 더 복잡한 특징 추출
#   → 풀링                       : 크기 축소
#   → Flatten                   : 2차원 특징맵을 1차원 벡터로 펼침
#   → 완전연결층(FullyConnection) : 내적 + bias
#   → Softmax                   : 점수를 확률(합=1)로 변환
#   → 출력 (분류 결과)

In [ ]:
# 객체 탐지: 이미지 안의 물체마다 "무엇인지(클래스) + 어디 있는지(박스 좌표) + 얼마나 확신하는지(신뢰도)"를 각각 출력하는 것. 분류는 답 1개, 탐지는 답 여러 개(개수 불특정)라는 게 핵심 차이
# IoU (Intersection over Union): 예측한 박스랑 실제 박스가 얼마나 겹치는지 재는 지표(겹친 넓이/합친 넓이)인데, 계산이 간단하고 어떤 탐지 모델을 쓰든 공통으로 쓰이는 기본 단위
# NMS (Non-Maximum Suppression): 같은 물체에 중복으로 그려진 여러 박스 중 제일 확신도 높은 것만 남기고 나머지 지우는 후처리 알고리즘 — 어떤 탐지 모델(YOLO든 뭐든)이든 다 이 단계를 거침
# mAP (mean Average Precision): "탐지 모델의 성능을 재는 지표"